# Data modeling 2: star schema design

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-02-star-schema-design.ipynb)

Lesson 1 built a fact table and two plain dimensions keyed by source IDs. A production star schema adds the
pieces that keep it correct as sources and history change: warehouse-owned **surrogate keys**, a **date
dimension**, dimensions that play several **roles**, a **junk dimension** for leftover flags, and
**conformed dimensions** shared by several facts.

By the end of the lesson the shop has a complete star: `fact_sales` at order-line grain and `fact_payment`
at payment grain, sharing `dim_date` and `dim_customer`.

**Prerequisites:** [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb).

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order) and `src_payments` (one row per payment; an order can have several): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
""")
con.sql("show tables").show()

## Surrogate keys

A **natural key** identifies something in a source system: `customer_id = 101` in the CRM. A **surrogate key**
is an integer the warehouse assigns to each dimension *row*, with no meaning outside the warehouse. Facts store
the surrogate key.

Why not just use the natural key?

- **History.** When a dimension keeps history (lesson 3), one customer has several rows. Each row needs its own
  key, and the natural key repeats.
- **Several sources.** The CRM's customer 101 and the billing system's customer 101 may be different
  customers. A surrogate key separates identity in the warehouse from identity in any one source.
- **Unknown members.** A fact can arrive before its customer exists, or with no customer at all. A reserved
  row such as `-1 'Unknown'` lets every fact join to something instead of disappearing from inner joins.
- **Stability.** Sources recycle IDs, change their format, or are replaced. Facts keyed by surrogates do not
  care.

Generate surrogates with a sequence or identity column (most warehouses) or `row_number()` over the new rows.
Some teams use a **hash key** (`md5(source || '|' || natural_key)`) instead: it is deterministic, so loads
can run in parallel without looking keys up, at the cost of a wider key.

**Where it matters**
- Every dimension that keeps history or merges more than one source.
- Fact loads: the load looks up each natural key and stores the surrogate it maps to (or the unknown member).

**Watch out for**
- Regenerating surrogates on every full rebuild with `row_number()`: keys move, and facts saved elsewhere point at the wrong rows.
- Inner joins on a fact whose key could not be resolved: rows vanish. Map them to the unknown member instead.

### Example

Build `dim_customer` with a surrogate key and an unknown member, then resolve a batch of incoming orders, one of
which names a customer the warehouse has not seen.

In [ ]:
con.sql("""
create sequence customer_sk_seq start 1;
create table dim_customer (
    customer_sk int primary key, customer_id int, customer_name varchar, city varchar, segment varchar
);
insert into dim_customer values (-1, null, 'Unknown', 'Unknown', 'Unknown');  -- the reserved unknown member
insert into dim_customer
select nextval('customer_sk_seq'), customer_id, name, city, segment from src_customers order by customer_id;
""")
con.sql("select * from dim_customer order by customer_sk").show()

In [ ]:
con.sql("""
with incoming(order_id, customer_id) as (values (2001, 103), (2002, 999), (2003, null))
select i.order_id, i.customer_id,
       d.customer_sk                     as inner_join_key,
       coalesce(d.customer_sk, -1)       as resolved_key
from incoming i left join dim_customer d on d.customer_id = i.customer_id
order by i.order_id
""").show()

**What to notice:** Customer 103 resolves to surrogate key 3. Customer 999 is not in the dimension yet and order 2003 has no customer:
an inner join would drop both orders and sales would quietly shrink. Mapping them to `-1` keeps every order in
the totals, reported under "Unknown" until the customer arrives (lesson 7 repairs these rows).

## The date dimension

Every fact has at least one date, and every report groups by some part of it: month, weekday, fiscal quarter,
holiday or not. A **date dimension** has one row per calendar day with those attributes precomputed, so the
rules (when the fiscal year starts, which days are holidays, how weeks are numbered) live in one table instead
of in every query.

Conventions:

- The key is often an integer in `YYYYMMDD` form (`20260105`). It is readable, sorts correctly and partitions
  well. A plain `date` key works too.
- Generate the rows for the whole range you need, including days with no activity.
- Add reserved rows for "unknown" and "not applicable" dates (for an order that has not shipped yet).

Because the dimension has a row for every day, it also answers "which days had *no* sales": start from the
dimension and left join the fact.

**Where it matters**
- Fiscal calendars, retail 4-4-5 calendars, holidays and working days: business rules that must be consistent.
- Reports that need a row per period even when nothing happened (zero-filled charts, gap detection).

**Watch out for**
- Time of day does not belong in the date dimension (it would have 86,400 times more rows); use a separate time dimension or keep the timestamp on the fact.
- Time zones: decide which zone's calendar day a timestamp belongs to before taking its date.

### Example

Generate January and February 2026 with a fiscal year that starts on April 1, then count orders per weekday
including weekdays with no orders.

In [ ]:
con.sql("""
create table dim_date as
select cast(strftime(d, '%Y%m%d') as int)                       as date_key,
       cast(d as date)                                          as date,
       year(d)                                                  as year,
       month(d)                                                 as month,
       strftime(d, '%b %Y')                                     as month_name,
       dayname(d)                                               as weekday,
       isodow(d) >= 6                                           as is_weekend,
       case when month(d) >= 4 then year(d) + 1 else year(d) end as fiscal_year  -- FY2026 runs Apr 2025 to Mar 2026
from range(date '2026-01-01', date '2026-03-01', interval 1 day) t(d)
union all
select -1, null, null, null, 'Not yet', 'Not yet', null, null    -- for milestones that have not happened
""")
con.sql("select * from dim_date where date_key in (20260101, 20260102, 20260103, -1) order by date_key").show()

In [ ]:
con.sql("""
select d.weekday, count(distinct o.order_id) as orders
from dim_date d
left join src_orders o on cast(o.order_ts as date) = d.date
where d.date_key <> -1
group by d.weekday, isodow(d.date)
order by isodow(d.date)
""").show()

**What to notice:** January 1, 2026 is a Thursday in fiscal year 2026; January 3 is the first weekend day. In the weekday count,
Thursday and Sunday show 0 orders. A query that groups `src_orders` alone could not show those rows, because there
is no order to group. Starting from the dimension guarantees every weekday appears.

## Role-playing dimensions

One fact row can refer to the same dimension more than once, in different roles: an order has an order date,
a ship date and a delivery date; a flight has an origin and a destination airport. You keep **one** physical
dimension and join it once per role, under a different alias (or through views named `dim_order_date`,
`dim_ship_date`). Each role gets its own key column on the fact.

**Where it matters**
- Fulfilment facts (ordered, shipped, delivered), insurance claims (incident, reported, settled), travel (origin, destination).

**Watch out for**
- Joining the dimension once and filtering both dates against it: you silently require them to be the same day.
- Reports that say "by month" without saying which date: orders by order month and by delivery month differ.

### Example

Count delivered orders by the month they were placed and by the month they were delivered.

In [ ]:
con.sql("""
create table fact_order as
select o.order_id,
       coalesce(c.customer_sk, -1)                                       as customer_sk,
       cast(strftime(o.order_ts, '%Y%m%d') as int)                       as order_date_key,
       coalesce(cast(strftime(o.delivered_ts, '%Y%m%d') as int), -1)     as delivered_date_key,
       o.channel, o.status, o.shipping_fee
from src_orders o left join dim_customer c using (customer_id)
""")
con.sql("""
select od.month_name as month,
       count(*)                                      as placed,
       (select count(*) from fact_order f2 join dim_date dd on dd.date_key = f2.delivered_date_key
        where dd.month_name = od.month_name)         as delivered
from fact_order f
join dim_date od on od.date_key = f.order_date_key      -- dim_date in the role of order date
group by od.month_name, od.month
order by od.month
""").show()

In [ ]:
con.sql("""
select f.order_id, od.date as ordered, dd.date as delivered, dd.month_name as delivered_month
from fact_order f
join dim_date od on od.date_key = f.order_date_key
join dim_date dd on dd.date_key = f.delivered_date_key   -- the same table in the role of delivery date
where f.order_id in (1004, 1006, 1008)
order by f.order_id
""").show()

**What to notice:** Four orders were placed in January but only three delivered then: order 1004 was placed on January 28 and
delivered on February 2, so it counts in January as an order and in February as a delivery. Orders 1006 and 1008
have not been delivered and join to the `-1` "Not yet" row instead of disappearing.

## Star or snowflake

In a **star**, each dimension is one denormalized table: `dim_product` carries `category` and `department`
as plain columns, repeated on every product. A **snowflake** normalizes them into their own tables
(`dim_product → dim_category → dim_department`), like an operational schema.

| | Star | Snowflake |
|---|---|---|
| Joins per query | one per dimension | one per level |
| Storage | repeats attribute text | stores it once |
| Changing a category name | update many product rows | update one row |
| BI tools and users | simple | more to understand |

Columnar warehouses compress repeated values well, so the storage argument for snowflaking is weak, and the
extra joins cost performance and clarity. Default to a star. Snowflake a part of a dimension when it is
large, shared by several dimensions, or maintained separately (a geography hierarchy used by customer and
store dimensions, for example).

**Where it matters**
- Deciding how to model hierarchies inside dimensions (product → category → department, city → region → country).

**Watch out for**
- Snowflaking by habit from operational modeling: it adds joins without adding information.

### Example

The same question on a star and on a snowflaked product dimension.

In [ ]:
con.sql("""
create table dim_product as  -- star: one table, hierarchy flattened onto each product
select row_number() over (order by sku) as product_sk, sku, product_name, category, department from src_products;

create table dim_department as
select row_number() over (order by department) as department_sk, department from (select distinct department from src_products);
create table dim_category as
select row_number() over (order by category) as category_sk, category, department_sk
from (select distinct category, department from src_products) join dim_department using (department);
create table dim_product_sf as  -- snowflake: the product points at its category, the category at its department
select p.product_sk, p.sku, p.product_name, c.category_sk from dim_product p join dim_category c using (category);
""")
con.sql("""
select 'star' as model, p.department, count(*) as products
from dim_product p group by all
union all
select 'snowflake', d.department, count(*)
from dim_product_sf p join dim_category c using (category_sk) join dim_department d using (department_sk)
group by all
order by model desc, department
""").show()

**What to notice:** Both models give 4 products per department. The star needs no join to reach `department`; the snowflake needs
two. The snowflake stores "Kitchen" once instead of four times, a saving a columnar engine already makes with
compression.

## Junk dimensions

After the main dimensions are designed, a few low-cardinality flags and codes are often left over: channel,
order status, gift-wrap yes or no, payment type. Putting each on the fact as text widens a very long table;
giving each its own tiny dimension adds joins. A **junk dimension** holds every combination that occurs, one
row each, and the fact carries a single key to it.

It works when the attributes are few and their combinations are few (here 2 channels × 5 statuses at most).
Attributes with many values, or that change independently of the others, belong in real dimensions.

**Where it matters**
- Order flags and codes, transaction types, small status combinations.

**Watch out for**
- Combining attributes with many values: the number of combinations explodes and the table stops being small.

### Example

Build a junk dimension from the channel and status combinations that occur.

In [ ]:
con.sql("""
create table dim_order_profile as
select row_number() over (order by channel, status) as order_profile_key, channel, status
from (select distinct channel, status from src_orders);
""")
con.sql("select * from dim_order_profile order by order_profile_key").show()

**What to notice:** Only 5 of the 10 possible combinations occur, so the fact needs a single small integer instead of two text
columns. New combinations are added as they appear, like any other dimension member.

## Conformed dimensions and drill-across

A warehouse has many fact tables, one per business process: sales, payments, shipments, returns. They are
useful together only if they share dimensions. A **conformed dimension** is one dimension table (same keys,
same attribute values) used by several facts. With `dim_customer` and `dim_date` conformed, "sales and
payments by month and segment" is a meaningful comparison.

The **bus matrix** plans this: business processes as rows, dimensions as columns, a mark where a process uses
a dimension. Each column marked more than once must be conformed.

| Process (fact) | Date | Customer | Product | Order profile | Payment method |
|---|---|---|---|---|---|
| Sales (`fact_sales`) | ✓ | ✓ | ✓ | ✓ | |
| Payments (`fact_payment`) | ✓ | ✓ | | | ✓ |
| Fulfilment (`fact_order`) | ✓ (several roles) | ✓ | | ✓ | |

To combine two facts, **drill across**: aggregate each fact to the shared dimension attributes separately,
then join the small results on those attributes. Never join two fact tables row to row: they have different
grains, and the join multiplies rows (lesson 4).

**Where it matters**
- Any report that compares processes: sales vs payments, orders vs returns, signups vs activations.
- Planning a warehouse: the bus matrix shows which dimensions to build first because most facts need them.

**Watch out for**
- Two teams each building their own `dim_customer` with different keys or segment rules: their numbers will never reconcile.
- Joining facts directly on `order_id` or `customer_id` instead of drilling across.

### Example

Build the two facts on conformed keys and compare sales with payments by month. Order 1003 was paid partly in
January and partly in February, and order 1006 was paid in March.

In [ ]:
con.sql("""
create table fact_sales as
select f.order_id, l.line_no,
       f.order_date_key, f.customer_sk, p.product_sk, j.order_profile_key,
       l.qty as quantity, l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l
join fact_order f using (order_id)
join dim_product p using (sku)
join dim_order_profile j on j.channel = f.channel and j.status = f.status
where f.status <> 'cancelled';

create table fact_payment as
select p.payment_id, p.order_id, cast(strftime(p.paid_ts, '%Y%m%d') as int) as paid_date_key,
       f.customer_sk, p.method, p.amount
from src_payments p join fact_order f using (order_id);
""")
con.sql("""
with sales as (
    select d.month_name, d.month, sum(s.net_amount) as net_sales
    from fact_sales s join dim_date d on d.date_key = s.order_date_key group by all
),
payments as (
    select d.month_name, d.month, sum(p.amount) as paid
    from fact_payment p left join dim_date d on d.date_key = p.paid_date_key group by all
)
select coalesce(s.month_name, p.month_name, 'outside dim_date') as month, s.net_sales, p.paid
from sales s full join payments p using (month_name, month)
order by coalesce(s.month, p.month)
""").show()

**What to notice:** Each fact was summed on its own and then joined month to month, so neither total is inflated. Payments in a
month differ from sales because payment includes shipping fees and follows its own timing. The March payment
for order 1006 shows as `outside dim_date`: `dim_date` only covers January and February. A dimension that does
not cover the facts' dates is a real bug; extending `dim_date` well into the future is the usual fix.

### Example

The finished star in use: net sales by department and segment for web orders.

In [ ]:
con.sql("""
select p.department, c.segment, sum(s.quantity) as units, sum(s.net_amount) as net_sales
from fact_sales s
join dim_product p using (product_sk)
join dim_customer c using (customer_sk)
join dim_order_profile j using (order_profile_key)
where j.channel = 'web'
group by all
order by all
""").show()

**What to notice:** One fact and three dimensions, each joined on a single integer key; filters and groups come only from
dimensions. That regular shape is what makes a star easy for people and BI tools to query, and easy for engines
to optimize.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A source system reuses customer IDs after an account is closed. How does a surrogate key help?

**A.** Facts point at the warehouse's surrogate key, not the source ID. The reused ID becomes a new dimension row with a new surrogate, so old facts keep pointing at the old customer.

**Q2.** Why should a fact load map a missing customer to `-1` instead of leaving the key null?

**A.** A null key drops the row from inner joins and makes "Unknown" impossible to group by. The `-1` member keeps the fact in every total and makes the problem visible.

**Q3.** A dashboard shows more orders in February than the orders team expects. What is the first thing you ask?

**A.** Which date it groups by. Counting by delivery or payment date moves orders between months compared with counting by order date.

**Q4.** When would you snowflake part of a dimension?

**A.** When that part is large, shared by several dimensions (a geography used by customers and stores) or maintained by a different process. Otherwise keep the star.

**Q5.** How do you report sales and payments by customer segment without double counting?

**A.** Aggregate each fact by segment on its own, then join the two small results on segment. Both facts must use the same conformed `dim_customer`.

## Key takeaways

- Facts store warehouse-owned surrogate keys; dimensions map natural keys to them.
- Reserve unknown and not-applicable members so unresolved facts stay in the totals.
- A date dimension holds calendar and fiscal rules once and has a row for every day, including quiet ones.
- Join one physical dimension several times for its roles, and say which date a report uses.
- Prefer stars to snowflakes; put leftover flags in a junk dimension.
- Conform shared dimensions across facts, plan them with a bus matrix, and drill across instead of joining facts.

**Next:** Continue with [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb).